In [3]:
import pandas as pd
import pickle
from functools import partial
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,balanced_accuracy_score
from src.utils.funciones_pipeline import pipeline,cols_nulos_wrapper,relleno_nulos_wrapper
import os

#os.chdir(os.path.dirname(__file__)) #cambio de directorio

#target
def construir_target(df, target):

    import re

    t = target[target["Country"].notna()].copy()
    t = t[~t["Country"].astype(str).str.fullmatch(r"\d{4}")]
    t = t[t["Country"].str.lower() != "total"]
    t = t[~t["Country"].astype(str).str.startswith("1/")]

    def extract_years(x):
        if pd.isna(x):
            return []
        return [int(y) for y in re.findall(r"\d{4}", str(x))]

    crisis_cols = {
        "banking_start": "Systemic Banking Crisis (starting date)",
        "currency": "Currency Crisis",
        "sovereign_debt": "Sovereign Debt Crisis (year)",
        "debt_restruct": "Sovereign Debt Restructuring (year)"
    }

    rows = []
    for crisis_type, col in crisis_cols.items():
        tmp = t[["Country", col]].copy()
        tmp["year"] = tmp[col].apply(extract_years)
        tmp = tmp.explode("year").dropna(subset=["year"])
        tmp["crisis_type"] = crisis_type
        rows.append(tmp[["Country", "crisis_type", "year"]])

    events_long = pd.concat(rows, ignore_index=True).drop_duplicates()

    events_wide = (
        events_long.assign(value=1)
        .pivot_table(index=["Country", "year"],
                     columns="crisis_type",
                     values="value",
                     aggfunc="max",
                     fill_value=0)
        .reset_index()
    )

    cols = ["banking_start", "currency", "debt_restruct", "sovereign_debt"]
    events_wide["Crisis"] = (events_wide[cols].sum(axis=1) > 0).astype(int)

    crisis_start = events_wide.loc[events_wide.Crisis > 0, ["Country", "year"]]
    crisis_start["year_pred"] = crisis_start["year"] - 1
    crisis_start["crisis_target"] = 1

    keys = crisis_start[["Country", "year_pred"]].drop_duplicates()
    keys = keys.rename(columns={"Country": "Country Name", "year_pred": "year"})

    df["crisis_target"] = 0
    df = df.merge(keys.assign(crisis_target=1),
                  on=["Country Name", "year"],
                  how="left",
                  suffixes=("", "_y"))

    df["crisis_target"] = df["crisis_target_y"].fillna(df["crisis_target"]).astype(int)
    df = df.drop(columns=["crisis_target_y"])

    return df


#train
def train_model():

    #datos
    df = pd.read_excel("./src/data_sample/Datos_paises_despivotados.xlsx")
    target = pd.read_excel("./src/data_sample/TARGET.xlsx")

    # target
    df = construir_target(df, target)

    #preprocesado antes de guardar elmodelo
    p = pipeline(cols_nulos_wrapper)
    df = p(df)

    p = pipeline(partial(relleno_nulos_wrapper, how="mean"))
    df = p(df)

    #cols finales
    cols_final = [
        'Deposit interest rate (%)',
        'Broad money (% of GDP)',
        'Exports of goods and services (current US$)',
        'Imports of goods and services (current US$)',
        'External debt stocks (% of GNI)',
        'Total debt service (% of exports of goods, services and primary income)',
        'GDP growth (annual %)',
        'GDP per capita growth (annual %)',
        'Foreign direct investment, net inflows (% of GDP)',
        'Inflation, consumer prices (annual %)'
    ]

    X = df[cols_final]
    y = df["crisis_target"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, shuffle=False)

    #Yeo-Johnson+RobustScaler
    yeo_pipeline = Pipeline([
        ("yeo", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", RobustScaler())
    ])

    preprocessing = ColumnTransformer(
        [("Process_Numeric_Log", yeo_pipeline, cols_final)],
        remainder="passthrough"
    )


    # scale_pos_weight
    n_pos = (y == 1).sum()
    n_neg = (y == 0).sum()
    spw = n_neg / n_pos

    #XGBoost
    model = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        random_state=42,
        scale_pos_weight=spw
    )

    #pipeline
    pipe = Pipeline([
        ("preprocess", preprocessing),
        ("model", model)
    ])

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=4,
                                scoring="balanced_accuracy")
    print("Balanced Accuracy CV:", cv_scores.mean())
    #entrenar train
    pipe.fit(X_train, y_train)


    #preds
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    #metricas
    print("Métricas de ML_CRISIS_PREDICTION")
    print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))

    #entrenar con todo
    pipe.fit(X, y)


    #gyardar modelo y pipeline completo
    with open("new_modelo_xgb.pkl", "wb") as f:
        pickle.dump(pipe, f)
        
    print("Modelo guardado correctamente")


In [5]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\prueba_API_Crisis_Prediction'

In [6]:
from model import train_model

In [7]:
train_model()

Unnamed: 0                                                                 0.0
Country Name                                                               0.0
Country Code                                                               0.0
year                                                                       0.0
Bank nonperforming loans to total gross loans (%)                          0.0
Broad money (% of GDP)                                                     0.0
Current account balance (% of GDP)                                         0.0
Deposit interest rate (%)                                                  0.0
Domestic credit to private sector (% of GDP)                               0.0
Domestic credit to private sector by banks (% of GDP)                      0.0
Exports of goods and services (% of GDP)                                   0.0
Exports of goods and services (current US$)                                0.0
External debt stocks (% of GNI)                     

In [9]:
import pickle

# Ruta a tu modelo
model_path = "new_modelo_xgb.pkl"

# Cargar modelo
with open(model_path, "rb") as f:
    model = pickle.load(f)

model

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Process_Numeric_Log', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [10]:
# Extraer columnas que espera el modelo desde el ColumnTransformer
expected_cols = model.named_steps["preprocess"].transformers_[0][2]

expected_cols

['Deposit interest rate (%)',
 'Broad money (% of GDP)',
 'Exports of goods and services (current US$)',
 'Imports of goods and services (current US$)',
 'External debt stocks (% of GNI)',
 'Total debt service (% of exports of goods, services and primary income)',
 'GDP growth (annual %)',
 'GDP per capita growth (annual %)',
 'Foreign direct investment, net inflows (% of GDP)',
 'Inflation, consumer prices (annual %)']